In [1]:
import os
import polars as pl
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
df = pl.read_parquet("../data/data_cleaned.parquet")
df.head()

race,gender,age,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,examide,citoglipton,insulin,glyburide-metformin,change,diabetesMed,label,admission_type,admission_source,discharge_disposition
str,str,f64,i64,str,str,i64,i64,i64,i64,i64,i64,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str
"""Caucasian""","""Female""",5.0,1,"""Other""","""Pediatrics-Endocrinology""",41,0,1,0,0,0,"""endocrine, nutritional and met…","""?""","""?""",1,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",0,"""NULL""","""Physician Referral""","""Not Mapped"""
"""Caucasian""","""Female""",15.0,3,"""Other""","""?""",59,0,18,0,0,0,"""endocrine, nutritional and met…","""endocrine, nutritional and met…","""endocrine, nutritional and met…",9,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Up""","""No""","""Ch""","""Yes""",1,"""Emergency""","""Emergency Room""","""Discharged to home"""
"""AfricanAmerican""","""Female""",25.0,2,"""Other""","""?""",11,5,13,2,0,1,"""complications of pregnancy, ch…","""endocrine, nutritional and met…","""Persons encountering health se…",6,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",0,"""Emergency""","""Emergency Room""","""Discharged to home"""
"""Caucasian""","""Male""",35.0,2,"""Other""","""?""",44,1,16,0,0,0,"""infectious and parasitic disea…","""endocrine, nutritional and met…","""diseases of the circulatory sy…",7,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Up""","""No""","""Ch""","""Yes""",0,"""Emergency""","""Emergency Room""","""Discharged to home"""
"""Caucasian""","""Male""",45.0,1,"""Other""","""?""",51,0,8,0,0,0,"""neoplasms""","""neoplasms""","""endocrine, nutritional and met…",5,"""None""","""None""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""No""","""No""","""No""","""No""","""Steady""","""No""","""Ch""","""Yes""",0,"""Emergency""","""Emergency Room""","""Discharged to home"""


In [3]:
df.describe()

statistic,race,gender,age,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,examide,citoglipton,insulin,glyburide-metformin,change,diabetesMed,label,admission_type,admission_source,discharge_disposition
str,str,str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str
"""count""","""101766""","""101766""",101766.0,101766.0,"""101766""","""101766""",101766.0,101766.0,101766.0,101766.0,101766.0,101766.0,"""101766""","""101766""","""101766""",101766.0,"""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""","""101766""",101766.0,"""101766""","""101766""","""101766"""
"""null_count""","""0""","""0""",0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""0""","""0""","""0"""
"""mean""",null,null,65.967022,4.395987,null,null,43.095641,1.33973,16.021844,0.369357,0.197836,0.635566,null,null,null,7.422607,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.460881,null,null,null
"""std""",null,null,15.940838,2.985108,null,null,19.674362,1.705807,8.127566,1.267265,0.930472,1.262863,null,null,null,1.9336,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.49847,null,null,null
"""min""","""AfricanAmerican""","""Female""",5.0,1.0,"""Other""","""?""",1.0,0.0,1.0,0.0,0.0,0.0,"""?""","""?""","""?""",1.0,"""High""",""">7""","""Down""","""Down""","""Down""","""Down""","""Down""","""Down""","""Down""","""Down""","""Down""","""No""","""No""","""Down""","""Down""","""Ch""","""No""",0.0,"""Elective""","""Clinic Referral""","""Admitted as an inpatient to th…"
"""25%""",null,null,55.0,2.0,null,null,31.0,0.0,10.0,0.0,0.0,0.0,null,null,null,6.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,null,null
"""50%""",null,null,65.0,4.0,null,null,44.0,1.0,15.0,0.0,0.0,0.0,null,null,null,8.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,null,null,null
"""75%""",null,null,75.0,6.0,null,null,57.0,2.0,20.0,0.0,0.0,1.0,null,null,null,9.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.0,null,null,null
"""max""","""Other""","""Unknown/Invalid""",95.0,14.0,"""Uninsured""","""Urology""",132.0,6.0,81.0,42.0,76.0,21.0,"""symptoms, signs, and ill-defin…","""symptoms, signs, and ill-defin…","""symptoms, signs, and ill-defin…",16.0,"""Normal""","""Norm""","""Up""","""Up""","""Up""","""Up""","""Up""","""Up""","""Up""","""Up""","""Up""","""No""","""No""","""Up""","""Up""","""No""","""Yes""",1.0,"""Urgent""","""Transfer from hospital inpt/sa…","""Still patient or expected to r…"


In [4]:
X = df.drop(['label'])
y = df['label']

ordinal_cols = ['max_glu_serum', 'A1Cresult']

max_glu_categories    = [['None', 'Normal', 'High']]
A1Cresult_categories  = [['None', 'Norm', '>7', '>8']]

all_ordinal_categories = max_glu_categories + A1Cresult_categories


# Polars equivalent of select_dtypes
numerical_cols  = [col for col, dtype in zip(X.columns, X.dtypes) 
                   if dtype in [pl.Int64, pl.Float64, pl.Int32, pl.Float32]]
categorical_cols = [col for col, dtype in zip(X.columns, X.dtypes) 
                    if dtype in [pl.Utf8, pl.String, pl.Categorical, pl.Boolean]]
categorical_cols = [c for c in categorical_cols if c not in ordinal_cols]


In [5]:
# Split Test and Training data
# Convert polars DataFrame and Series to pandas DataFrame and Series
X_train, X_test, y_train, y_test = train_test_split(X.to_pandas(),
                                                    y.to_pandas(),
                                                    test_size=0.3,
                                                    random_state=42,
                                                    stratify=y)

In [6]:
pl.DataFrame(X_train).describe()

statistic,race,gender,age,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,examide,citoglipton,insulin,glyburide-metformin,change,diabetesMed,admission_type,admission_source,discharge_disposition
str,str,str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""count""","""71236""","""71236""",71236.0,71236.0,"""71236""","""71236""",71236.0,71236.0,71236.0,71236.0,71236.0,71236.0,"""71236""","""71236""","""71236""",71236.0,"""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236""","""71236"""
"""null_count""","""0""","""0""",0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0"""
"""mean""",null,null,65.942922,4.397187,null,null,43.119055,1.338031,16.017547,0.370655,0.197892,0.636293,null,null,null,7.422258,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",null,null,15.914404,2.986965,null,null,19.72728,1.706129,8.128062,1.274228,0.938027,1.261743,null,null,null,1.932817,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""AfricanAmerican""","""Female""",5.0,1.0,"""Other""","""?""",1.0,0.0,1.0,0.0,0.0,0.0,"""?""","""?""","""?""",1.0,"""High""",""">7""","""Down""","""Down""","""Down""","""Down""","""Down""","""Down""","""Down""","""Down""","""Down""","""No""","""No""","""Down""","""Down""","""Ch""","""No""","""Elective""","""Clinic Referral""","""Admitted as an inpatient to th…"
"""25%""",null,null,55.0,2.0,null,null,31.0,0.0,10.0,0.0,0.0,0.0,null,null,null,6.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,null,65.0,4.0,null,null,44.0,1.0,15.0,0.0,0.0,0.0,null,null,null,8.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,null,75.0,6.0,null,null,57.0,2.0,20.0,0.0,0.0,1.0,null,null,null,9.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""Other""","""Unknown/Invalid""",95.0,14.0,"""Uninsured""","""Urology""",132.0,6.0,81.0,40.0,76.0,21.0,"""symptoms, signs, and ill-defin…","""symptoms, signs, and ill-defin…","""symptoms, signs, and ill-defin…",16.0,"""Normal""","""Norm""","""Up""","""Up""","""Up""","""Up""","""Up""","""Up""","""Up""","""Up""","""Up""","""No""","""No""","""Up""","""Up""","""No""","""Yes""","""Urgent""","""Transfer from hospital inpt/sa…","""Still patient or expected to r…"


In [7]:
X_train.head()

,race,gender,age,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,...,rosiglitazone,examide,citoglipton,insulin,glyburide-metformin,change,diabetesMed,admission_type,admission_source,discharge_disposition
51913,AfricanAmerican,Female,75.0,5,Public,?,25,3,21,2,...,No,No,No,Up,No,Ch,Yes,Not Available,Physician Referral,Discharged/transferred to SNF
24583,Caucasian,Male,65.0,3,Other,?,35,2,20,1,...,No,No,No,No,No,No,No,Elective,Physician Referral,Discharged to home
53974,AfricanAmerican,Female,85.0,3,Public,?,56,2,11,0,...,No,No,No,Steady,No,No,Yes,Emergency,Emergency Room,Discharged to home
2843,Caucasian,Male,75.0,7,Other,InternalMedicine,75,0,11,0,...,Steady,No,No,Steady,No,Ch,Yes,Emergency,Emergency Room,Discharged to home
86023,Caucasian,Male,65.0,4,Public,Emergency/Trauma,1,0,17,0,...,No,No,No,Down,No,Ch,Yes,Emergency,Emergency Room,Discharged/transferred to another rehab fac in...


In [8]:
# Build preprocessing pipeline
preprocessor = ColumnTransformer(transformers=[
    ('num', RobustScaler(), numerical_cols),
    ('ord', OrdinalEncoder(categories=all_ordinal_categories), ordinal_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

# Fit on train (dev) set only
X_train_scaled = preprocessor.fit_transform(X_train)

# Transform all 3 dev, validation, and test sets
X_test_scaled  = preprocessor.transform(X_test)

In [9]:
os.makedirs('../data/scaled', exist_ok=True)

feature_names = preprocessor.get_feature_names_out().tolist()

# --- Scaled X files ---
pl.DataFrame(X_train_scaled,  schema=feature_names).write_parquet('../data/scaled/X_train.parquet')
pl.DataFrame(X_test_scaled, schema=feature_names).write_parquet('../data/scaled/X_test.parquet')

# --- y files ---
pl.DataFrame({'label': y_train.to_list()}).write_parquet('../data/scaled/y_train.parquet')
pl.DataFrame({'label': y_test.to_list()}).write_parquet('../data/scaled/y_test.parquet')

In [10]:
# os.makedirs('../data', exist_ok=True)

# # Use unscaled pandas DataFrames directly
# dev_df  = pl.from_pandas(X_dev)
# val_df  = pl.from_pandas(X_val)
# test_df = pl.from_pandas(X_test)

# # Add target column
# dev_df  = dev_df.with_columns(pl.Series('label', y_dev.to_list()))
# val_df  = val_df.with_columns(pl.Series('label', y_val.to_list()))
# test_df = test_df.with_columns(pl.Series('label', y_test.to_list()))

# # Combine dev + val into full train
# train_df = pl.concat([dev_df, val_df])

# # Save all four
# train_df.write_parquet('../data/train.parquet')
# dev_df.write_parquet('../data/dev.parquet')
# val_df.write_parquet('../data/val.parquet')
# test_df.write_parquet('../data/test.parquet')

# print(f"train.parquet : {train_df.shape}")
# print(f"dev.parquet   : {dev_df.shape}")
# print(f"val.parquet   : {val_df.shape}")
# print(f"test.parquet  : {test_df.shape}")
# print(f"\nSaved to: {os.path.abspath('../data/')}")

In [11]:
files = ['train', 'test']

for f in files:
    df = pl.read_parquet(f'../data/{f}.parquet')
    print(f'=== {f}.parquet {df.shape} ===')
    print(df.describe())
    print()

=== train.parquet (81412, 37) ===
shape: (9, 38)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ race      ┆ gender    ┆ age       ┆ … ┆ admission ┆ admission ┆ discharge ┆ readmitt │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ _type     ┆ _source   ┆ _disposit ┆ ed       │
│ str       ┆ str       ┆ str       ┆ f64       ┆   ┆ ---       ┆ ---       ┆ ion       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ str       ┆ str       ┆ ---       ┆ f64      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ str       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 81412     ┆ 81412     ┆ 81412.0   ┆ … ┆ 81412     ┆ 81412     ┆ 81412     ┆ 81412.0  │
│ null_coun ┆ 0         ┆ 0         ┆ 0.0       ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 0.0      │
│ t         ┆           ┆           ┆     